# Kakunin + Google Antigravity SDK Compliance Playground

This notebook demonstrates how to register agent compliance certificates and integrate scope validation hooks with Google Antigravity AI agents.

### Step 1: Install Dependencies
First, we'll install `kakunin`, `google-antigravity`, and `python-dotenv`.

In [ ]:
!pip install kakunin google-antigravity python-dotenv

### Step 2: Configure Environment Keys
Set your Kakunin and Gemini API keys below.

In [ ]:
import os
import getpass

os.environ["KAK_API_KEY"] = getpass.getpass("Enter Kakunin API Key (kak_live_...): ")
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key (AIzaSy...): ")

### Step 3: Run the Agent Integration Demo
Now we'll run a script to register the agent with Kakunin, issue its compliance certificate, bind tool scopes, and run the agent.

In [ ]:
import asyncio
from google.antigravity import Agent, LocalAgentConfig
from kakunin import Kakunin
from kakunin.exceptions import ScopeViolationError
from kakunin.integrations.google_antigravity import get_kakunin_hooks

# Define mock financial tools
def query_market_prices(symbol: str) -> str:
    """Query current prices for a given ticker symbol."""
    print(f"[Tool Executing] query_market_prices: {symbol}")
    return f"Latest price for {symbol}: $150.00"

def execute_market_trade(symbol: str, amount: int) -> str:
    """Execute a market buy order for a symbol."""
    print(f"[Tool Executing] execute_market_trade: Buying {amount} of {symbol}")
    return f"Successfully executed trade: Buy {amount} shares of {symbol}"

async def run_compliance_demo():
    async with Kakunin(api_key=os.environ["KAK_API_KEY"]) as kakunin_client:
        # 1. Register agent
        agent_record = await kakunin_client.agents.create(
            name="NotebookTrader",
            model="gemini-3.5-flash",
            version="1.0.0",
            model_hash="sha256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
        )
        print(f"Registered Agent: {agent_record.id}")

        # 2. Issue certificate
        cert = await kakunin_client.agents.certify(agent_record.id)
        print(f"Issued Certificate Serial: {cert.serial_number}")

        # 3. Create compliance hooks with scope mapping
        hooks = get_kakunin_hooks(
            kakunin=kakunin_client,
            agent_id=agent_record.id,
            tool_scopes_mapping={
                "query_market_prices": ["market.read"],
                "execute_market_trade": ["trade.execute"]
            }
        )

        # 4. Initialize Antigravity agent
        config = LocalAgentConfig(
            model="gemini-3.5-flash",
            system_instructions="You are a helpful assistant with financial tools.",
            tools=[query_market_prices, execute_market_trade],
            hooks=hooks
        )

        async with Agent(config=config) as agent:
            print("\n--- Query Price ---")
            try:
                res = await agent.chat("Check price of GOOG")
                print(f"Response: {res}")
            except ScopeViolationError as e:
                print(f"Compliance block: {e}")

            print("\n--- Execute Trade ---")
            try:
                res = await agent.chat("Buy 5 shares of GOOG")
                print(f"Response: {res}")
            except ScopeViolationError as e:
                print(f"Compliance block: {e}")

await run_compliance_demo()